# 02 — Train / Validation / Test Split

**Input:** IMDB 50K reviews (from `01-EDA`).
**Output:** three CSVs — `train.csv`, `val.csv`, `test.csv` — saved to `../data/`.

This split is created **once** and reused by every downstream model (TF-IDF
baseline, DistilBERT, and the Claude eval). A single canonical split is what
makes the three-way comparison fair — all models are scored on the identical
held-out reviews.

## Setup

`SEED` is defined once and reused everywhere (split, training, sampling) so the
entire pipeline is reproducible: same seed → same split, every run.

In [1]:
from sklearn.model_selection import train_test_split
import pandas as pd
SEED = 132

## Load and de-duplicate

EDA found **418 duplicate reviews**. We remove them from the full pool *before*
splitting — if we split first, copies of the same review could land on both
sides of the train/test line, leaking test data into training and inflating our
scores. Dedupe first, then split the clean pool.

In [2]:
# Dropping the duplicates first
df = pd.read_csv('../data/IMDB Dataset.csv')
df = df.drop_duplicates(subset='review').reset_index(drop=True)

## Split: 70 / 10 / 15 stratified, two-stage

Target: **70% train / 10% validation / 20% test** (of the de-duplicated data).

`train_test_split` only does a two-way split, so we call it twice:
1. Hold out 20% as the **test** set (never touched until final evaluation).
2. Split the remaining 80% into train and val.

On the second call, `test_size` is a fraction of the 80% pool, not the original —
so val = `0.10 / 0.80` = 0.125.

`stratify=sentiment` preserves the 50/50 class balance in every split;
`random_state=SEED` makes the partition deterministic.

In [3]:
# Spliting the dataset into Train, Val and Test sets

Train_val, test = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df['sentiment'])

# Deviding validation set from the Train_val set

val_fraction = 0.10/0.80  # 10% of the original dataset is 12.5% of the Train_val set
train, val = train_test_split(Train_val, test_size=val_fraction, random_state=SEED, stratify=Train_val['sentiment'])

print(len(train), len(val), len(test))

# Saving the datasets into CSV files
train.to_csv('../data/train.csv', index=False)
val.to_csv('../data/val.csv', index=False)
test.to_csv('../data/test.csv', index=False)

34706 4959 9917


In [4]:
assert set(train['review']) & set(test['review']) == set(), "Overlap between train and test!"
assert set(train['review']) & set(val['review'])  == set(), "Overlap between train and val!"
print("No overlap — splits are clean.")

No overlap — splits are clean.


## Summary

- De-duplicated (418 removed), then split **70/10/15** with a fixed seed.
- Stratified on sentiment → class balance preserved across all three sets.
- No train/val/test overlap (asserted above).
- Saved to `../data/{train,val,test}.csv` for every downstream model to share.

**Next:** `03` — TF-IDF + Logistic Regression baseline, evaluated on `test.csv`.